<a href="https://colab.research.google.com/github/urvisrikm/Financial-Data-Analysis/blob/main/Financial-Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#importing the libraries which I will be using
import pandas as pd
import numpy as np

import os
if not os.path.exists('transactions.csv'):
    data = {
        'Date': ['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05', '2023-01-08', '2023-01-10'],
        'Description': ['Salary', 'Rent Payment', 'Amazon Shop', 'Grocery Store', 'Starbucks', 'Dinner Out', 'LARGE WIRE TRANSFER'],
        'Amount': [5000.00, -1500.00, -75.50, -120.00, -5.50, -60.00, -25000.00]
    }
    pd.DataFrame(data).to_csv('transactions.csv', index=False)

# Load the data
df = pd.read_csv('transactions.csv')

# Basic keyword tagging for categories
def get_category(text):
    t = text.lower()
    if 'salary' in t: return 'Income'
    if 'rent' in t: return 'Bills'
    if 'amazon' in t or 'shop' in t: return 'Discretionary'
    if 'grocery' in t or 'starbucks' in t: return 'Essentials'
    return 'Misc'

df['Category'] = df['Description'].apply(get_category)

# Statistical Fraud Check
# Here I will be using 2 standard deviations to catch outliers
outflows = df[df['Amount'] < 0]['Amount'].abs()
limit = outflows.mean() + (2 * outflows.std())

# Flagging anything over the statistical limit
df['Risk_Level'] = df['Amount'].apply(
    lambda x: 'CRITICAL' if abs(x) > limit and x < 0 else 'NORMAL'
)

print("--- JPMC AUDIT REPORT ---")
print(f"Calculated Fraud Limit: ${limit:.2f}")
print("\nSpending by Category:")
print(df.groupby('Category')['Amount'].sum())

print("\nSecurity Alerts (Flagged Transactions):")
alerts = df[df['Risk_Level'] == 'CRITICAL']
if not alerts.empty:
    print(alerts[['Date', 'Description', 'Amount']])
else:
    print("No critical risks found.")


--- JPMC AUDIT REPORT ---
Calculated Fraud Limit: $19542.21

Spending by Category:
Category
Bills            -1500.0
Discretionary     -275.5
Essentials        -125.5
Income            5000.0
Misc            -25660.0
Name: Amount, dtype: float64

Security Alerts (Flagged Transactions):
         Date                   Description   Amount
9  2023-01-10  Large Fraudulent Transaction -25000.0
